In [1]:
import pandas as pd
import os 
import glob
import re

# INPUTS (to be modified according to contract, quarter and language)
contract = '0044'
quarter = "2026 Q1"
chinese = "1"                   # "1" means data is in Chinese

# prior_to_change = "1"           # "1" means that if is prior to the further breakdown of the ad-supported stuff. So any report prior to 2024 should have "1"

outputdirectory = '../../50 KM Group/Royalties/Statements/Karen/_output/'

base_dir = f'../../50 KM Group/Royalties/Statements/Karen/Tencent - TME/Contract_{contract}/{quarter}/as supplied by TME/'
outputfilename = f"../../50 KM Group/Royalties/Statements/Karen/_output/TME_{contract}_{quarter}.xlsx"
lookup_platform = '../../50 KM Group/Royalties/Statements/Karen/Tencent - TME/support/lookup_platform.csv'
lookup_mv_album = '../../50 KM Group/Royalties/Statements/Karen/Tencent - TME/support/lookup_album_mv.csv'  
lookup_fx = '../../50 KM Group/Royalties/Statements/Karen/Tencent - TME/support/lookup_fx.csv' 

year = int(quarter.split()[0])

def find_date_range(filename):
    # Regular expression to match the date range format "YYYYMMDD-YYYYMMDD"
    pattern = r'\d{8}-\d{8}'
    # Search for the pattern in the filename
    match = re.search(pattern, filename)
    if match:
        return match.group()
    else:
        return None

subdirs_pattern = os.path.join(base_dir, '202*')
subdirs = list(set(glob.glob(subdirs_pattern)))

def read_and_combine_files(base_dir: str, keyword_file: str):
    base_dir.sort()
    df_all = []

    for subdir_per_month in base_dir:
        print(f'Accessing folder: {subdir_per_month}')
        df_list = []

        for files_in_one_month in os.listdir(subdir_per_month):
            if files_in_one_month.startswith(keyword_file):
                file_string = subdir_per_month + "/" + files_in_one_month
                df = pd.read_excel(file_string, sheet_name=0)
                print(file_string)
                if chinese == "1":
                    df['结算期间'] = find_date_range(file_string)
                else:
                    df['Period'] = find_date_range(file_string)
                df_list.append(df)
        if df_list:
            result_per_month = pd.concat(df_list, ignore_index=True)
            df_all.append(result_per_month)
    if df_all:
        final_df = pd.concat(df_all, ignore_index=True)
        return final_df
    else:
        return pd.DataFrame()  # Return an empty DataFrame if no relevant files were found

def merge(df1, df2, col):
    print(f"\nMerging on '{col}':")
    empty_cells = df1[col].isna().sum()
    print(f"There are a total of {empty_cells} rows that have no entry in {col}")
    df1.loc[:, col] = df1[col].fillna('n/a')
    #df1.fillna({col: 'n/a'}, inplace=True)
    df_merged = pd.merge(df1, df2, on=col, how='left')
    new_columns = df_merged.columns.difference(df1.columns)
    first_new_col = new_columns[0] if not new_columns.empty else None
    empty_cells2 = df_merged[first_new_col].isna().sum() if first_new_col else 0
    diff_empty = empty_cells2 - empty_cells
    if diff_empty == 0:
        print(f"Merging of column {col} was successful")
    else:
        print(f"Merging with issues. There are a total of {diff_empty} cells that could not be matched (see 'match_issues_{df1}_{col}.xlsx').")
        empty_rows = df_merged[df_merged[first_new_col].isna()]
        col=col.replace('/','')
        empty_rows.to_excel(f"{outputdirectory}/match_issues_{col}.xlsx", engine='openpyxl', index=False)
    print(f"The new dataframe has {df_merged.shape[0]} rows and {len(df_merged.columns)} columns.")
    return df_merged 

def add_columns_and_modify_columns_to_single_df(df_input:pd.DataFrame):
    df = df_input.copy()
    df.rename(columns={'License Fees - Total': 'Fee'}, inplace=True)
    df['Units'] = df['Sales_IOS'] + df['Sales_Others']
    df['UPC'] = pd.to_numeric(df['UPC'], errors='coerce')
    df['Composing Right End Date'] = df['Composing Right End Date'].astype(str)
    df['Composing Right Start Date'] = df['Composing Right Start Date'].astype(str)
    df['Product']='Purchase'
    return df

def add_columns_and_modify_columns_to_song_df(df_input:pd.DataFrame):
    df = df_input.copy()
    df['License Fees - Subscription Music Service(Senior)'] = pd.to_numeric(df['License Fees - Subscription Music Service(Senior)'], errors='coerce')
    df['License Fees - Subscription Music Service(Senior)'] = df['License Fees - Subscription Music Service(Senior)'].astype(float)
    return df

def add_columns_and_modify_columns_to_mv_df(df_input:pd.DataFrame):
    df = df_input.copy()
    df.rename(columns={'License Fees - Total': 'Fee'}, inplace=True)
    df.rename(columns={'Start Time of Authorization': 'Recording Right Start Date'}, inplace=True)
    df.rename(columns={'End Time of Authorization': 'Recording Right End Date'}, inplace=True)
    df.rename(columns={'Share of Recording Right': 'Right Share of Recorder'}, inplace=True)
    df.rename(columns={'MV': 'Song'}, inplace=True)
    df.rename(columns={'License Fees - Free Music Service-Non-free Mode-Consumption of MV': 'Units'}, inplace=True)
    if 'License Fees for the Ad-Supported Service—Non-free Mode' in df.columns:
        df = df.drop('License Fees for the Ad-Supported Service—Non-free Mode', axis=1)
    df['Product']='MV'
    return df

def add_columns_and_modify_columns_to_digalbum_df(df_input:pd.DataFrame):
    df = df_input.copy()
    df.rename(columns={'License Fees - Total': 'Fee'}, inplace=True)
    df['Units'] = df['Sales_IOS'] + df['Sales_Others']
    df = df.drop('Chargeable Digital Album Service',axis=1)
    df = df.drop('原始版权公司', axis=1)
    df = df.drop('版权公司名',axis=1)
    df = df.drop('版权公司歌曲编码',axis=1)
    df = df.drop('邻接权授权有效期开始',axis=1)
    df = df.drop('邻接权授权有效期结束',axis=1)
    df['Product']='Digital Album'
    return df

def add_columns_and_modify_columns_to_k_df(df_input:pd.DataFrame):
    df = df_input.copy()
    df.rename(columns={'License Fees - Total': 'Fee'}, inplace=True)
    #df.rename(columns={'License Fees-Per Play-Consumption of Recording-Original Version': 'Units'}, inplace=True)
    df['Units'] = df['License Fees-Per Play-Consumption of Lyrics'] + df['License Fees-Per Play-Consumption of Composition'] + df['License Fees-Per Play-Consumption of Recording-Original Version']
    df = df.drop('License Fees-Per Play-Consumption of Recording-Original Version', axis=1)
    df = df.drop('License Fees-Per Play-Consumption of Lyrics', axis=1)
    df = df.drop('License Fees-Per Play-Consumption of Composition', axis=1)
    df = df.drop('License Fees-Per Play-Consumption of Recording-Karaoke Version Provided by Licensor', axis=1)
    df = df.drop('License Fees-Per Play-Consumption of Recording-Karaoke Version Processed by TME', axis=1)
    df = df.drop('License Fees-Per Play', axis=1)
    df['Product']='Karaoke'
    return df

def rename_single_columns(df_input:pd.DataFrame):
    df = df_input.copy()
    df = df.rename(columns={
 #   '结算期间' : 'Period',
 #   '平台' : 'Platform',
 #   '日期' : 'Date',
 #   '中央曲库歌曲ID' : 'TMEID',
 #   '专辑名' : 'Album',
 #   '歌曲名' : 'Song',
 #   '歌手名' : 'Artist',
 #   '词作者' : 'Lyricist',
 #   '曲作者' : 'Composer',
 #   '专辑UPC' : 'UPC',
 #   '歌曲ISRC' : 'ISRC',
 #   '版权公司专辑编码' : 'Licensor Album-Code',
 #   '版权公司歌曲编码' : 'Licensor Code',
 #   '版权公司名' : 'Licensor',
 #   '原始版权公司' : 'Label',
 #   '词授权有效期开始' : 'Lyrics Right Start Date',
 #   '词授权有效期结束' : 'Lyrics Right End Date',
 #   '曲授权有效期开始' : 'Composing Right Start Date',
 #   '曲授权有效期结束' : 'Composing Right End Date',
 #   '邻接权授权有效期开始' : 'Recording Right Start Date',
 #   '邻接权授权有效期结束' : 'Recording Right End Date',
 #   '词份额' : 'Right Share of Writer',
 #   '曲份额' : 'Right Share of Composer',
 #   '邻接权份额' : 'Right Share of Recorder',
 #   '歌曲付费状态' : 'Charge Type',
 #   '单价' : 'Retail Price',
 #   'IOS销量' : 'Sales_IOS',
 #   '非IOS销量' : 'Sales_Others',
 #   '单曲订购金额' : "Single Track's Perchase - Revenue",
 #   '单曲订购收入分成' : 'License Fees - Single Track Purchase',
 #   'CP分成收入' : 'License Fees - Total'
     '结算期间' : 'Period',
     '平台' : 'Platform',
    '日期' : 'Date',
    '中央曲库歌曲ID' : 'TMEID',
    '专辑名' : 'Album',
    '歌曲名' : 'Song',
    '歌手名' : 'Artist',
    '词作者' : 'Lyricist',
    '曲作者' : 'Composer',
    '专辑UPC' : 'UPC',
    '歌曲ISRC' : 'ISRC',
    '版权公司专辑编码' : 'Licensor Album-Code',
    '版权公司歌曲编码' : 'Licensor Code',
    '版权公司名' : 'Licensor',
    '原始版权公司' : 'Label',
    '词授权有效期开始' : 'Lyrics Right Start Date',
    '词授权有效期结束' : 'Lyrics Right End Date',
    '曲授权有效期开始' : 'Composing Right Start Date',
    '曲授权有效期结束' : 'Composing Right End Date',
    '邻接权授权有效期开始' : 'Recording Right Start Date',
    '邻接权授权有效期结束' : 'Recording Right End Date',
    '词份额' : 'Right Share of Writer',
    '曲份额' : 'Right Share of Composer',
    '邻接权份额' : 'Right Share of Recorder',
    '歌曲付费状态' : 'Charge Type',
    '单价' : 'Retail Price',
    'IOS销量' : 'Sales_IOS',
    '非IOS销量' : 'Sales_Others',
    '单曲订购金额' : "Single Track's Perchase - Revenue",
    '单曲订购收入分成' : 'License Fees - Single Track Purchase',
    'CP分成收入' : 'License Fees - Total',
    '币种' : 'Currency',
    '汇率' : 'Exchange Rate',
    'CP分成收入(CNY)' : 'License Fee(CNY)'
    })
    return df

def rename_song_columns(df_input:pd.DataFrame):
    df = df_input.copy()
    df = df.rename(columns={
 #   '结算期间' : 'Period',
 #   '平台' : 'Platform',
 #   '中央曲库歌曲ID' : 'TMEID',
 #   '歌曲名' : 'Song',
 #   '歌曲ISRC' : 'ISRC',
 #   '歌手名' : 'Artist',
 #   '词作者' : 'Lyricist',
 #   '曲作者' : 'Composer',
 #   '专辑名' : 'Album',
 #   '专辑UPC' : 'UPC',
 #   '版权公司名' : 'Licensor',
 #   '原始版权公司' : 'Label',
 #   '词授权有效期开始' : 'Lyrics Right Start Date',
 #   '词授权有效期结束' : 'Lyrics Right End Date',
 #   '曲授权有效期开始' : 'Composing Right Start Date',
 #   '曲授权有效期结束' : 'Composing Right End Date',
 #   '邻接权授权有效期开始' : 'Recording Right Start Date',
 #   '邻接权授权有效期结束' : 'Recording Right End Date',
 #   '版权公司专辑编码' : 'Licensor Album-Code',
 #   '版权公司歌曲编码' : 'Licensor Code',
 #   '词份额' : 'Right Share of Writer',
 #   '曲份额' : 'Right Share of Composer',
 #   '邻接权份额' : 'Right Share of Recorder',
 #   '歌曲付费状态' : 'Charge Type',
 #   '广告收入分成-免模-使用量' : '(unclear)',
 #   '广告收入分成-非免模-使用量' : 'License Fees - Free Music Service-Free Mode-Number of Content Used',
  #  '歌曲驱动金额' : 'License Fees - Free Music Service-Non-free Mode-Number of Content Used',
  #  '基本包月收入分成-使用量' : 'Subscription Music Service(Basic)',
  #  '高级包月收入分成-使用量' : 'Subscription Music Service(Senior)',
  #  '打榜收入' : 'MuCoin & Gift',
  #  '广告收入分成-免模' : 'License Fees for the Ad-Supported Service—Free Mode',
  #  '广告收入分成-非免模' : 'License Fees for the Ad-Supported Service—Non-free Mode',
  #  '基本包月收入分成' : 'License Fees - Subscription Music Service(Basic)',
  #  '高级包月收入分成' : 'License Fees - Subscription Music Service(Senior)',
  #  '打榜收入分成' : 'License Fees - MuCoin & Gift',
  #  'CP分成收入' : 'License Fees - Total' 
    '结算期间' : 'Period',
    '平台' : 'Platform',
    '中央曲库歌曲ID' : 'TMEID',
    '歌曲名' : 'Song',
    '歌曲ISRC' : 'ISRC',
    '歌手名' : 'Artist',
    '词作者' : 'Lyricist',
    '曲作者' : 'Composer',
    '专辑名' : 'Album',
    '专辑UPC' : 'UPC',
    '版权公司名' : 'Licensor',
    '原始版权公司' : 'Label',
    '词授权有效期开始' : 'Lyrics Right Start Date',
    '词授权有效期结束' : 'Lyrics Right End Date',
    '曲授权有效期开始' : 'Composing Right Start Date',
    '曲授权有效期结束' : 'Composing Right End Date',
    '邻接权授权有效期开始' : 'Recording Right Start Date',
    '邻接权授权有效期结束' : 'Recording Right End Date',
    '版权公司专辑编码' : 'Licensor Album-Code',
    '版权公司歌曲编码' : 'Licensor Code',
    '词份额' : 'Right Share of Writer',
    '曲份额' : 'Right Share of Composer',
    '邻接权份额' : 'Right Share of Recorder',
    '歌曲付费状态' : 'Charge Type',
    '广告收入分成-免模-使用量' : 'License Fees - Free Music Service-Free Mode-Number of Content Used',
    '广告收入分成-非免模-使用量' : 'License Fees - Free Music Service-Non-free Mode-Number of Content Used',
    '基本包月收入分成-使用量' : 'Subscription Music Service(Basic)',
    '高级包月收入分成-使用量' : 'Subscription Music Service(Senior)',
    '打榜收入' : 'MuCoin & Gift',
    '广告收入分成-免模' : 'License Fees for the Ad-Supported Service—Free Mode',
    '广告收入分成-非免模' : 'License Fees for the Ad-Supported Service—Non-free Mode',
    '基本包月收入分成' : 'License Fees - Subscription Music Service(Basic)',
    '高级包月收入分成' : 'License Fees - Subscription Music Service(Senior)',
    '打榜收入分成' : 'License Fees - MuCoin & Gift',
    'CP分成收入' : 'License Fees - Total',
    '币种' : 'Currency',
    '汇率' : 'Exchange Rate',
    'CP分成收入(CNY)' : 'License Fee(CNY)'
    })
    return df

def rename_aiting_columns(df_input:pd.DataFrame):
    df = df_input.copy()
    df = df.rename(columns={
    '结算期间' : 'Period',
    '平台' : 'Platform',
    '渠道商' : 'Distributor',
    '中央曲库ID' : 'TMEID',
    '歌曲名' : 'Song',
    '歌曲ISRC' : 'ISRC',
    '歌手名' : 'Artist',
    '词作者' : 'Lyricist',
    '曲作者' : 'Composer',
    '专辑名' : 'Album',
    '专辑UPC' : 'UPC',
    '版权公司名' : 'Licensor',
    '原始版权公司' : 'Label',
    '词授权有效期开始' : 'Lyrics Right Start Date',
    '词授权有效期结束' : 'Lyrics Right End Date',
    '曲授权有效期开始' : 'Composing Right Start Date',
    '曲授权有效期结束' : 'Composing Right End Date',
    '邻接权授权有效期开始' : 'Recording Right Start Date',
    '邻接权授权有效期结束' : 'Recording Right End Date',
    '版权公司专辑编码' : 'Licensor Album-Code',
    '版权公司歌曲编码' : 'Licensor Code',
    '词份额' : 'Right Share of Writer',
    '曲份额' : 'Right Share of Composer',
    '邻接权份额' : 'Right Share of Recorder',
    '歌曲付费状态' : 'Charge Type',
    '广告收入分成-使用量' : 'Free Music Service',
    '包月收入分成-使用量' : 'Consumption - Subscription Music Service',
    '广告收入分成' : 'License Fees - Free Music Service',
    '包月收入分成' : 'Licensee Fees - Subscription Music Service',
    'CP分成收入' : 'License Fees - Total',
    '币种' : 'Currency',
    '汇率' : 'Exchange Rate',
    'CP分成收入(CNY)' : 'License Fee(CNY)'
    })
    return df

def rename_mv_columns(df_input:pd.DataFrame):
    df = df_input.copy()
    df = df.rename(columns={
    '结算期间' : 'Period',
    '平台' : 'Platform',
    '中央曲库MVID' : 'MVID',
    'MV名' : 'MV',
    '歌手名' : 'Artist',
    '歌曲ISRC' : 'ISRC',
    '专辑UPC' : 'UPC',
    '版权公司名' : 'Licensor',
    '原始版权公司' : 'Label',
    '授权有效期开始' : 'Start Time of Authorization',
    '授权有效期结束' : 'End Time of Authorization',
    '邻接权比例' : 'Share of Recording Right',
    '广告收入分成-非免模-MV使用量' : 'License Fees - Free Music Service-Non-free Mode-Consumption of MV',
    '广告收入分成-非免模' : 'License Fees for the Ad-Supported Service—Non-free Mode',
    'CP分成收入' : 'License Fees - Total',
    '币种' : 'Currency',
    '汇率' : 'Exchange Rate',
    'CP分成收入(CNY)' : 'License Fee(CNY)'
    })
    return df

def rename_digalbum_columns(df_input:pd.DataFrame):
    df = df_input.copy()
    df = df.rename(columns={
    '结算期间' : 'Period',
    '平台' : 'Platform',
    '日期' : 'Date',
    '中央曲库专辑ID' : 'Album_ID',
    '专辑名' : 'Album',
    '歌手名' : 'Artist',
    '专辑UPC' : 'UPC',
    '版权公司专辑编码' : 'Licensor Album-Code',
    '词份额' : 'Right Share of Writer',
    '曲份额' : 'Right Share of Composer',
    '邻接权份额' : 'Right Share of Recorder',
    '单价' : 'Retail Price',
    'IOS销量' : 'Sales_IOS',
    '非IOS销量' : 'Sales_Others',
    '数字专辑销售金额' : 'Total Sales Amount of Digital Album',
    '数字专辑销售收入分成' : 'Chargeable Digital Album Service',
    'CP分成收入' : 'License Fees - Total',
    '币种' : 'Currency',
    '汇率' : 'Exchange Rate',
    'CP分成收入(CNY)' : 'License Fee(CNY)',
    '中央曲库歌曲ID' : 'TMEID',
    '曲作者' : 'Composer',
    '曲授权有效期开始' : 'Composing Right Start Date',
    '曲授权有效期结束' : 'Composing Right End Date',
    '歌曲ISRC' : 'ISRC',
    '歌曲名' : 'Song',
    '词作者' : 'Lyricist',
    '词授权有效期开始' : 'Lyrics Right Start Date',
    '词授权有效期结束' : 'Lyrics Right End Date'
    })
    return df

def rename_k_columns(df_input:pd.DataFrame):
    df = df_input.copy()
    df = df.rename(columns={
    '结算期间' : 'Period',
    '平台' : 'Platform',
    '中央曲库歌曲ID' : 'TMEID',
    '歌曲名' : 'Song',
    '歌曲ISRC' : 'ISRC',
    '歌手名' : 'Artist',
    '词作者' : 'Lyricist',
    '曲作者' : 'Composer',
    '专辑名' : 'Album',
    '专辑UPC' : 'UPC',
    '版权公司名' : 'Licensor',
    '原始版权公司' : 'Label',
    '词授权有效期开始' : 'Lyrics Right Start Date',
    '词授权有效期结束' : 'Lyrics Right End Date',
    '曲授权有效期开始' : 'Composing Right Start Date',
    '曲授权有效期结束' : 'Composing Right End Date',
    '邻接权授权有效期开始' : 'Recording Right Start Date',
    '邻接权授权有效期结束' : 'Recording Right End Date',
    '版权公司专辑编码' : 'Licensor Album-Code',
    '版权公司歌曲编码' : 'Licensor Code',
    '词份额' : 'Right Share of Writer',
    '曲份额' : 'Right Share of Composer',
    '邻接权份额' : 'Right Share of Recorder',
    '按次分成-词使用量' : 'License Fees-Per Play-Consumption of Lyrics',
    '按次分成-曲使用量' : 'License Fees-Per Play-Consumption of Composition',
    '按次分成-邻接权使用量-原版音源' : 'License Fees-Per Play-Consumption of Recording-Original Version',
    '按次分成-邻接权使用量-版权方提供伴奏' : 'License Fees-Per Play-Consumption of Recording-Karaoke Version Provided by Licensor',
    '按次分成-邻接权使用量-依据版权方提供音源制作伴奏' : 'License Fees-Per Play-Consumption of Recording-Karaoke Version Processed by TME',
    '按次分成' : 'License Fees-Per Play',
    'CP分成收入' : 'License Fees - Total',
    '币种' : 'Currency',
    '汇率' : 'Exchange Rate',
    'CP分成收入(CNY)' : 'License Fee(CNY)'
    })
    return df


def fill_missing_exchange_rates(main_df, Lookup_fx):
    """
    Fills missing 'exchange rate' and 'currency' values in main_df using data from Lookup_fx.

    Parameters:
        main_df (pd.DataFrame): The main DataFrame with columns 'platform', 'period', 'currency', 
                                'exchange rate', 'Fee (local currency)', and 'Fee (CNY)'.
        Lookup_fx (pd.DataFrame): The lookup DataFrame with columns 'platform', 'period', 
                                  'currency', and 'exchange rate'.

    Returns:
        pd.DataFrame: The updated main_df with missing 'exchange rate' and 'currency' filled in.
    """
    # Merge the two datasets on 'platform', 'period', and 'currency'
    merged_df = pd.merge(
        main_df,
        Lookup_fx,
        on=['Platform', 'Period'],
        how='left',  # Use 'left' to keep all rows from main_df
        suffixes=('', '_lookup')  # Add suffix to columns from Lookup_fx
    )

    #print(main_df.columns)
    #print(Lookup_fx.columns)
    #print(merged_df.columns)

    # Fill missing 'exchange rate' and 'currency' in main_df with values from Lookup_fx. 
    merged_df['Exchange Rate'].fillna(merged_df['Exchange Rate_lookup'], inplace=True)
    merged_df['Currency'].fillna(merged_df['Currency_lookup'], inplace=True)

    # Drop the extra columns added during the merge
    merged_df.drop(columns=['Exchange Rate_lookup', 'Currency_lookup','License Fee(CNY)'], inplace=True)

    #if 'Fee (CNY)' in merged_df.columns:
    merged_df.rename(columns={'Fee': 'Fee (local currency)'}, inplace=True)
    merged_df['Fee'] = merged_df['Fee (local currency)'] * merged_df['Exchange Rate']
    merged_df['Fee'].fillna(merged_df['Fee (local currency)'], inplace=True)
   

    # Return the updated DataFrame
    return merged_df


In [2]:
df_single = read_and_combine_files(base_dir = subdirs, keyword_file="single")
if chinese == "1":
        print(df_single.columns)
        df_single = rename_single_columns(df_single)
        print(df_single.columns)
df_single = add_columns_and_modify_columns_to_single_df(df_input = df_single)
df_single.head()

print(f"The combined DataFrame has {df_single.shape[0]} rows and {len(df_single.columns)} columns.")


Accessing folder: ../../50 KM Group/Royalties/Statements/Karen/Tencent - TME/Contract_0044/2026 Q1/as supplied by TME/2026 01
../../50 KM Group/Royalties/Statements/Karen/Tencent - TME/Contract_0044/2026 Q1/as supplied by TME/2026 01/single_outside_detail_CON02-TME00-20260127-0044_kg_325113751_20260101-20260131-0407154649.xlsx
../../50 KM Group/Royalties/Statements/Karen/Tencent - TME/Contract_0044/2026 Q1/as supplied by TME/2026 01/single_outside_detail_CON02-TME00-20260127-0044_kw_325113751_20260101-20260131-0407154406.xlsx
../../50 KM Group/Royalties/Statements/Karen/Tencent - TME/Contract_0044/2026 Q1/as supplied by TME/2026 01/single_outside_detail_CON02-TME00-20260127-0044_qq_325113751_20260101-20260131-0407154222.xlsx
Accessing folder: ../../50 KM Group/Royalties/Statements/Karen/Tencent - TME/Contract_0044/2026 Q1/as supplied by TME/2026 02
../../50 KM Group/Royalties/Statements/Karen/Tencent - TME/Contract_0044/2026 Q1/as supplied by TME/2026 02/single_outside_detail_CON02-TME

In [3]:
df_song = read_and_combine_files(base_dir = subdirs, keyword_file="song")
if chinese == "1":
        df_song = rename_song_columns(df_song)
df_song = add_columns_and_modify_columns_to_song_df(df_input = df_song)
df_song.head()

print(f"The combined DataFrame has {df_song.shape[0]} rows and {len(df_song.columns)} columns.")

Accessing folder: ../../50 KM Group/Royalties/Statements/Karen/Tencent - TME/Contract_0044/2026 Q1/as supplied by TME/2026 01
../../50 KM Group/Royalties/Statements/Karen/Tencent - TME/Contract_0044/2026 Q1/as supplied by TME/2026 01/song_outside_detail_CON02-TME00-20260127-0044_qq_325113751_20260101-20260131-0407152927.xlsx
../../50 KM Group/Royalties/Statements/Karen/Tencent - TME/Contract_0044/2026 Q1/as supplied by TME/2026 01/song_outside_detail_CON02-TME00-20260127-0044_jooxth_325113751_20260101-20260131-0407152218.xlsx
../../50 KM Group/Royalties/Statements/Karen/Tencent - TME/Contract_0044/2026 Q1/as supplied by TME/2026 01/song_outside_detail_CON02-TME00-20260127-0044_jooxid_325113751_20260101-20260131-0407151306.xlsx
../../50 KM Group/Royalties/Statements/Karen/Tencent - TME/Contract_0044/2026 Q1/as supplied by TME/2026 01/song_outside_detail_CON02-TME00-20260127-0044_kw_325113751_20260101-20260131-0407151407.xlsx
../../50 KM Group/Royalties/Statements/Karen/Tencent - TME/Con

In [4]:
# Files "song" - part B 

def create_subset(df, common_columns, specific_columns):
    """
    Create a subset of the DataFrame using common and specific columns.
    
    :param df: The original DataFrame.
    :param common_columns: List of columns common to all subsets.
    :param specific_columns: List of columns specific to each subset.
    :return: A subset DataFrame with selected columns.
    """
    # Combine common columns with specific columns
    columns_to_select = common_columns + specific_columns
    # Return the subset DataFrame
    return df[columns_to_select]

# Function to process each subset DataFrame
def process_dataframe(df, rename_dict, product, product_detail=None):
    df = df.rename(columns=rename_dict)
    df['Product'] = product
    if product_detail:
        df['Product (Detail)'] = product_detail
    return df

# List of common columns
common_columns = [
    'Period',
    'Platform',
    'TMEID',
    'Song',
    'ISRC',
    'Artist',
    'Lyricist',
    'Composer',
    'Album',
    'UPC',
    'Licensor',
    'Label',
    'Lyrics Right Start Date',
    'Lyrics Right End Date',
    'Composing Right Start Date',
    'Composing Right End Date',
    'Recording Right Start Date',
    'Recording Right End Date',
    'Licensor Album-Code',
    'Licensor Code',
    'Right Share of Writer',
    'Right Share of Composer',
    'Right Share of Recorder',
    'Charge Type'
]

# Free Music Service - Free Mode
if year == 2024 or year == 2025 or year == 2026:
    subset_df_4_6_free1 = create_subset(
        df_song,
        common_columns,
        [
            'License Fees - Free Music Service-Free Mode-Number of Content Used',
            'License Fees for the Ad-Supported Service—Free Mode'
        ]
    )
else:
    subset_df_4_6_free1 = create_subset(
        df_song,
        common_columns,
        [
            'Free Music Service',
            'License Fees - Free Music Service'
        ]
    ) 


# Free Music Service - Non-free Mode
if year == 2024 or year == 2025 or year == 2026:
    subset_df_4_6_free2 = create_subset(
        df_song,
        common_columns,
        [
            'License Fees - Free Music Service-Non-free Mode-Number of Content Used',
            'License Fees for the Ad-Supported Service—Non-free Mode'
        ]
    )

# Subscription Music Service (Basic)
subset_df_4_6_subscription = create_subset(
    df_song,
    common_columns,
    [
        'Subscription Music Service(Basic)',
        'License Fees - Subscription Music Service(Basic)'
    ]
)

# Subscription Music Service (Senior)
subset_df_4_6_subscription_premium = create_subset(
    df_song,
    common_columns,
    [
        'Subscription Music Service(Senior)',
        'License Fees - Subscription Music Service(Senior)'
    ]
)

# MuCoin & Gift
subset_df_4_6_MUcoins = create_subset(
    df_song,
    common_columns,
    [
        'MuCoin & Gift',
        'License Fees - MuCoin & Gift'
    ]
)

# List of dictionaries with DataFrame information
if year == 2024 or year == 2025 or year == 2026: 
    dataframes_info = [
        {
            'df': subset_df_4_6_free1,
            'rename_dict': {'License Fees - Free Music Service-Free Mode-Number of Content Used': 'Units', 'License Fees for the Ad-Supported Service—Free Mode': 'Fee'},
            'product': 'Ad-supported',
            'product_detail': 'Free Music Service-Free Mode'
        },
        {
            'df': subset_df_4_6_free2,
            'rename_dict': {'License Fees - Free Music Service-Non-free Mode-Number of Content Used': 'Units', 'License Fees for the Ad-Supported Service—Non-free Mode': 'Fee'},
            'product': 'Ad-supported',
            'product_detail': 'Free Music Service-Non-free Mode'
        },
        {
            'df': subset_df_4_6_subscription,
            'rename_dict': {'Subscription Music Service(Basic)': 'Units', 'License Fees - Subscription Music Service(Basic)': 'Fee'},
            'product': 'Subscription (Basic)'
        },
        {
            'df': subset_df_4_6_subscription_premium,
            'rename_dict': {'Subscription Music Service(Senior)': 'Units', 'License Fees - Subscription Music Service(Senior)': 'Fee'},
            'product': 'Subscription (Premium)'
        },
        {
            'df': subset_df_4_6_MUcoins,
            'rename_dict': {'MuCoin & Gift': 'Units', 'License Fees - MuCoin & Gift': 'Fee'},
            'product': 'MU Coin'
        }
    ]
else: 
    dataframes_info = [
        {
            'df': subset_df_4_6_free1,
            'rename_dict': {'Free Music Service': 'Units', 'License Fees - Free Music Service': 'Fee'},
            'product': 'Ad-supported',
        },
        {
            'df': subset_df_4_6_subscription,
            'rename_dict': {'Subscription Music Service(Basic)': 'Units', 'License Fees - Subscription Music Service(Basic)': 'Fee'},
            'product': 'Subscription (Basic)'
        },
        {
            'df': subset_df_4_6_subscription_premium,
            'rename_dict': {'Subscription Music Service(Senior)': 'Units', 'License Fees - Subscription Music Service(Senior)': 'Fee'},
            'product': 'Subscription (Premium)'
        },
        {
            'df': subset_df_4_6_MUcoins,
            'rename_dict': {'MuCoin & Gift': 'Units', 'License Fees - MuCoin & Gift': 'Fee'},
            'product': 'MU Coin'
        }
    ]

# Process each DataFrame and store the result in a list
processed_dfs = [process_dataframe(info['df'], info['rename_dict'], info['product'], info.get('product_detail')) for info in dataframes_info]

# Convert 'Fee' to float specifically for MU Coin DataFrame
processed_dfs[-1]['Fee'] = processed_dfs[-1]['Fee'].astype(float)

# Concatenate all processed DataFrames
df_song_modified = pd.concat(processed_dfs, ignore_index=True)

print(f"The combined DataFrame has {df_song_modified.shape[0]} rows and {len(df_song_modified.columns)} columns.")

The combined DataFrame has 14700 rows and 28 columns.


In [5]:
df_aiting = read_and_combine_files(base_dir = subdirs, keyword_file="aiting")
if not df_aiting.empty:  
    if chinese == "1":
        df_aiting = rename_aiting_columns(df_aiting)
    df_aiting.head()
    print(f"The combined DataFrame has {df_aiting.shape[0]} rows and {len(df_aiting.columns)} columns.")

Accessing folder: ../../50 KM Group/Royalties/Statements/Karen/Tencent - TME/Contract_0044/2026 Q1/as supplied by TME/2026 01
../../50 KM Group/Royalties/Statements/Karen/Tencent - TME/Contract_0044/2026 Q1/as supplied by TME/2026 01/aiting_song_outside_detail_CON02-TME00-20260127-0044_325113751_20260101-20260131-0407151753.xlsx
Accessing folder: ../../50 KM Group/Royalties/Statements/Karen/Tencent - TME/Contract_0044/2026 Q1/as supplied by TME/2026 02
../../50 KM Group/Royalties/Statements/Karen/Tencent - TME/Contract_0044/2026 Q1/as supplied by TME/2026 02/aiting_song_outside_detail_CON02-TME00-20260127-0044_325113918_20260201-20260228-0407161124.xlsx
Accessing folder: ../../50 KM Group/Royalties/Statements/Karen/Tencent - TME/Contract_0044/2026 Q1/as supplied by TME/2026 03
../../50 KM Group/Royalties/Statements/Karen/Tencent - TME/Contract_0044/2026 Q1/as supplied by TME/2026 03/aiting_song_outside_detail_CON02-TME00-20260127-0044_325748431_20260301-20260331-0421232752.xlsx
The com

In [6]:
# Part B for "aiting" file

# Dictionary to define renaming and product details
columns_info = {
    'Free Music Service': {
        'rename': {'Free Music Service': 'Units', 'License Fees - Free Music Service': 'Fee'},
        'product': 'Ad-supported'
    },
    'Consumption - Subscription Music Service': {
        'rename': {'Consumption - Subscription Music Service': 'Units', 'Licensee Fees - Subscription Music Service': 'Fee'},
        'product': 'Subscription'
    }
}

# Function to process and rename columns, add 'Product' column
def process_and_label(df, column_key):
    df = df.rename(columns=columns_info[column_key]['rename'])
    df['Product'] = columns_info[column_key]['product']
    return df

# Process both DataFrames
subset_df_7_free = process_and_label(df_aiting[[
    'Period', 'Platform', 'Distributor', 'TMEID', 'Song', 'ISRC', 'Artist', 'Lyricist', 'Composer', 
    'Album', 'UPC', 'Licensor', 'Label', 'Lyrics Right Start Date', 'Lyrics Right End Date', 
    'Composing Right Start Date', 'Composing Right End Date', 'Recording Right Start Date', 
    'Recording Right End Date', 'Licensor Album-Code', 'Licensor Code', 'Right Share of Writer', 
    'Right Share of Composer', 'Right Share of Recorder', 'Charge Type', 'Free Music Service', 
    'License Fees - Free Music Service'
]], 'Free Music Service')

subset_df_7_subscription = process_and_label(df_aiting[[
    'Period', 'Platform', 'Distributor', 'TMEID', 'Song', 'ISRC', 'Artist', 'Lyricist', 'Composer', 
    'Album', 'UPC', 'Licensor', 'Label', 'Lyrics Right Start Date', 'Lyrics Right End Date', 
    'Composing Right Start Date', 'Composing Right End Date', 'Recording Right Start Date', 
    'Recording Right End Date', 'Licensor Album-Code', 'Licensor Code', 'Right Share of Writer', 
    'Right Share of Composer', 'Right Share of Recorder', 'Charge Type', 
    'Consumption - Subscription Music Service', 'Licensee Fees - Subscription Music Service'
]], 'Consumption - Subscription Music Service')

# Concatenate the processed DataFrames
df_aiting_modified = pd.concat([subset_df_7_free, subset_df_7_subscription], ignore_index=True)
#print(df_aiting_modified)


In [7]:
df_mv = read_and_combine_files(base_dir = subdirs, keyword_file="mv")
if not df_mv.empty:
    if chinese == "1":
        # print(df_mv)
        df_mv = rename_mv_columns(df_mv)
        # print(df_mv)
    df_mv = add_columns_and_modify_columns_to_mv_df(df_input = df_mv)
    df_mv_album = pd.read_csv(lookup_mv_album)
    df_mv = merge(df_mv, df_mv_album, 'Song')
    df_mv.head()
    print(f"The combined DataFrame has {df_mv.shape[0]} rows and {len(df_mv.columns)} columns.")


Accessing folder: ../../50 KM Group/Royalties/Statements/Karen/Tencent - TME/Contract_0044/2026 Q1/as supplied by TME/2026 01
../../50 KM Group/Royalties/Statements/Karen/Tencent - TME/Contract_0044/2026 Q1/as supplied by TME/2026 01/mv_outside_detail_ver_CON02-TME00-20260127-0044_qq_325113751_20260101-20260131-0408095525.xlsx
../../50 KM Group/Royalties/Statements/Karen/Tencent - TME/Contract_0044/2026 Q1/as supplied by TME/2026 01/mv_outside_detail_ver_CON02-TME00-20260127-0044_kg_325113751_20260101-20260131-0408095343.xlsx
../../50 KM Group/Royalties/Statements/Karen/Tencent - TME/Contract_0044/2026 Q1/as supplied by TME/2026 01/mv_outside_detail_ver_CON02-TME00-20260127-0044_kw_325113751_20260101-20260131-0408095321.xlsx
Accessing folder: ../../50 KM Group/Royalties/Statements/Karen/Tencent - TME/Contract_0044/2026 Q1/as supplied by TME/2026 02
../../50 KM Group/Royalties/Statements/Karen/Tencent - TME/Contract_0044/2026 Q1/as supplied by TME/2026 02/mv_outside_detail_ver_CON02-TME

In [8]:
df_digalbum = read_and_combine_files(base_dir = subdirs, keyword_file="digital")
if not df_digalbum.empty:
    if chinese == "1":
        df_digalbum = rename_digalbum_columns(df_digalbum)
    df_digalbum = add_columns_and_modify_columns_to_digalbum_df(df_input = df_digalbum)
    df_digalbum.head()
    print(f"The combined DataFrame has {df_digalbum.shape[0]} rows and {len(df_digalbum.columns)} columns.")

Accessing folder: ../../50 KM Group/Royalties/Statements/Karen/Tencent - TME/Contract_0044/2026 Q1/as supplied by TME/2026 01
../../50 KM Group/Royalties/Statements/Karen/Tencent - TME/Contract_0044/2026 Q1/as supplied by TME/2026 01/digital_album_outside_detail_CON02-TME00-20260127-0044_qq_325113751_20260101-20260131-0407153908.xlsx
../../50 KM Group/Royalties/Statements/Karen/Tencent - TME/Contract_0044/2026 Q1/as supplied by TME/2026 01/digital_album_outside_detail_CON02-TME00-20260127-0044_kw_325113751_20260101-20260131-0407154026.xlsx
../../50 KM Group/Royalties/Statements/Karen/Tencent - TME/Contract_0044/2026 Q1/as supplied by TME/2026 01/digital_album_outside_detail_CON02-TME00-20260127-0044_kg_325113751_20260101-20260131-0407154059.xlsx
Accessing folder: ../../50 KM Group/Royalties/Statements/Karen/Tencent - TME/Contract_0044/2026 Q1/as supplied by TME/2026 02
../../50 KM Group/Royalties/Statements/Karen/Tencent - TME/Contract_0044/2026 Q1/as supplied by TME/2026 02/digital_al

In [9]:
df_k = read_and_combine_files(base_dir = subdirs, keyword_file="k_outside")
if not df_k.empty:  
    if chinese == "1":
        df_k = rename_k_columns(df_k)
    df_k = add_columns_and_modify_columns_to_k_df(df_input = df_k)
    df_k.head()
    print(f"The combined DataFrame has {df_k.shape[0]} rows and {len(df_k.columns)} columns.")

Accessing folder: ../../50 KM Group/Royalties/Statements/Karen/Tencent - TME/Contract_0044/2026 Q1/as supplied by TME/2026 01
../../50 KM Group/Royalties/Statements/Karen/Tencent - TME/Contract_0044/2026 Q1/as supplied by TME/2026 01/k_outside_detail_CON02-TME00-20260127-0044_325113751_20260101-20260131-0407160249.xlsx
Accessing folder: ../../50 KM Group/Royalties/Statements/Karen/Tencent - TME/Contract_0044/2026 Q1/as supplied by TME/2026 02
../../50 KM Group/Royalties/Statements/Karen/Tencent - TME/Contract_0044/2026 Q1/as supplied by TME/2026 02/k_outside_detail_CON02-TME00-20260127-0044_325113918_20260201-20260228-0407165234.xlsx
Accessing folder: ../../50 KM Group/Royalties/Statements/Karen/Tencent - TME/Contract_0044/2026 Q1/as supplied by TME/2026 03
../../50 KM Group/Royalties/Statements/Karen/Tencent - TME/Contract_0044/2026 Q1/as supplied by TME/2026 03/k_outside_detail_CON02-TME00-20260127-0044_325748431_20260301-20260331-0422024413.xlsx
The combined DataFrame has 3657 rows 

In [10]:
# Function to align columns of two DataFrames by adding missing columns with NA values
def align_columns(df1, df2):
    # Get missing columns for both DataFrames
    missing_in_df1 = df2.columns.difference(df1.columns)
    missing_in_df2 = df1.columns.difference(df2.columns)
    
    # Add missing columns to each DataFrame
    for col in missing_in_df1:
        df1[col] = pd.NA
    for col in missing_in_df2:
        df2[col] = pd.NA
        
    return df1, df2

df_single, df_song_modified = align_columns(df_single, df_song_modified)
df_single_song = pd.concat([df_single, df_song_modified], ignore_index=True)

df_single_song, df_aiting_modified = align_columns(df_single_song, df_aiting_modified)
df_single_song_aiting = pd.concat([df_single_song, df_aiting_modified], ignore_index=True)

df_mv, df_single_song_aiting = align_columns(df_mv, df_single_song_aiting)
df_single_song_aiting_mv = pd.concat([df_mv, df_single_song_aiting], ignore_index=True)

df_digalbum, df_single_song_aiting_mv = align_columns(df_digalbum, df_single_song_aiting_mv)
df_single_song_aiting_mv_digalbum = pd.concat([df_digalbum, df_single_song_aiting_mv], ignore_index=True)

df_k, df_single_song_aiting_mv_digalbum = align_columns(df_k, df_single_song_aiting_mv_digalbum)
df_final1 = pd.concat([df_k, df_single_song_aiting_mv_digalbum], ignore_index=True)


# Assuming main_df and Lookup_fx are already defined
df_fx = pd.read_csv(lookup_fx)
df_final = fill_missing_exchange_rates(df_final1, df_fx)

# Add 'Contract' and 'Quarter' columns
df_final['Contract'] = contract
df_final['Quarter'] = quarter
df_platform = pd.read_csv(lookup_platform)
df_final = pd.merge(df_final, df_platform, on='Platform', how='left')
df_final = df_final.sort_index(axis=1)

print(f"The final DataFrame has {df_final.shape[0]} rows and {len(df_final.columns)} columns.")

print(f"Total fee: {df_final['Fee'].sum()}. Total units: {df_final['Units'].sum()}")



# Save the final DataFrame to Excel
df_final.to_excel(outputfilename, engine='openpyxl', index=True)


The final DataFrame has 24361 rows and 44 columns.
Total fee: 804003.4920529991. Total units: 118542945.0


/var/folders/r4/jcfnn_jn7d78r15gxq27mq2m0000gn/T/ipykernel_16085/674463924.py:16: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_single_song = pd.concat([df_single, df_song_modified], ignore_index=True)
/var/folders/r4/jcfnn_jn7d78r15gxq27mq2m0000gn/T/ipykernel_16085/674463924.py:19: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_single_song_aiting = pd.concat([df_single_song, df_aiting_modified], ignore_index=True)
/var/folders/r4/jcfnn_jn7d78r15gxq27mq2m0000gn/T/ipykernel_16085/674463924.p